# Data Migration from CSV files to PostgreSQL database

The following code is used to load  data from local `.csv` files directly into the database defined in the application.
***

## Database connection

In [4]:
import pandas as pd
import os
import sys
from dotenv import load_dotenv
from sqlalchemy import text

# Add parent directory to path (to enable imports from utils)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.db_handler import DatabaseHandler

In [5]:
# Load environment variables
env_path = os.path.join('..', '.env')
if not os.path.exists(env_path):
    env_path = os.path.join('..', '.env.example')
load_dotenv(env_path)

# Build database URL for local environment (using localhost instead of docker service name)
db_user = os.getenv('POSTGRES_USER')
db_password = os.getenv('POSTGRES_PASSWORD')
db_name = os.getenv('POSTGRES_DB')
db_port = os.getenv('POSTGRES_PORT')
db_host = 'localhost'

database_url = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

# Initialize database connection
db = DatabaseHandler(database_url=database_url)

---
## Bootcamps and living costs in Polish voivodeship cities
The following code is used to load raw data from  `.csv` files directly into the database defined in the application.

In [35]:
# Define paths to CSV files - assuming files are located in the project root folder
bootcamps_csv = '../examples/Bootcamps.xlsx'
living_costs_csv = '../examples/Living_costs.csv'

### Bootcamp Data Loading
This cell handles the extraction of data from Excel and its insertion into the PostgreSQL database.

In [36]:
def load_bootcamp_data(file_path, db_handler):
    """
    Reads bootcamp data from an Excel file and loads it into the database.

    Args:
        file_path (str): Path to the Excel file.
        db_handler: Object containing the SQLAlchemy engine.
    """
    try:
        # Note: pd.read_excel is used as per original logic for bootcamp data
        df_bootcamp = pd.read_excel(file_path)

        # Save to database in 'bootcamp_data' table
        # 'replace' will drop the table if it exists and recreate it
        df_bootcamp.to_sql('bootcamp_data', con=db_handler.engine, if_exists='replace', index=False)

        print(f"Successfully saved bootcamp data ({len(df_bootcamp)} rows) to table 'bootcamp_data'.")

    except FileNotFoundError:
        print(f"Critical Error: File '{file_path}' not found.")
    except Exception as e:
        print(f"An unexpected error occurred while loading bootcamp data: {e}")

# Execution
# load_bootcamp_data(bootcamps_csv, db)

Saved bootcamp data (27 rows) to table 'bootcamp_data'.
Saved living costs data (16 rows) to table 'living_costs_data'.


### Living Costs Data Loading
This cell handles the extraction of data from CSV and its insertion into the PostgreSQL database.

In [ ]:
def load_living_costs_data(file_path, db_handler):
    """
    Reads living costs data from a CSV file and loads it into the database.

    Args:
        file_path (str): Path to the CSV file.
        db_handler: Object containing the SQLAlchemy engine.
    """
    try:
        df_costs = pd.read_csv(file_path, sep=',')

        # Save to database in 'living_costs_data' table
        df_costs.to_sql('living_costs_data', con=db_handler.engine, if_exists='replace', index=False)

        print(f"Successfully saved living costs data ({len(df_costs)} rows) to table 'living_costs_data'.")

    except FileNotFoundError:
        print(f"Critical Error: File '{file_path}' not found.")
    except Exception as e:
        print(f"An unexpected error occurred while loading living costs data: {e}")

# Execution
# load_living_costs_data(living_costs_csv, db)

### Main Execution Orchestrator
In Jupyter, it is often useful to have a single cell that triggers all loading functions.

In [ ]:
def run_data_import_pipeline():
    """
    Orchestrates the loading of various data sources into the PostgreSQL database.
    """
    print("Starting Data Import Pipeline...")

    # 1. Process Bootcamp Data
    load_bootcamp_data(bootcamps_csv, db)

    # 2. Process Living Costs Data
    load_living_costs_data(living_costs_csv, db)

    print("Data Import Pipeline finished.")

# Trigger the pipeline
run_data_import_pipeline()

---
## Raw job postings data
The following code is used to load raw data from  `.csv` files directly into the database defined in the application.

### File Selection Logic
This part handles the GUI interaction.

In [11]:
def select_csv_files():
    """
    Opens a file dialog to select multiple CSV files.

    Returns:
        list: A list of selected file paths.
    """
    root = tk.Tk()
    root.attributes('-topmost', True)
    root.withdraw()

    file_paths = filedialog.askopenfilenames(
        title="Select CSV files with job postings",
        filetypes=[("CSV Files", "*.csv")]
    )
    root.destroy()
    return file_paths

file_paths = select_csv_files()

if not file_paths:
    print("No files selected. Operation cancelled.")
else:
    print(f"Selected {len(file_paths)} files.")


Usunięto 4698 zduplikowanych wierszy z samej paczki CSV (zostawiono najnowsze).
Łącznie 3314 unikalnych wierszy gotowych do zmergowania w bazie.
Sukces! Całość (3314 unikalnych wierszy) została poprawnie zmergowana z tabelą 'job_postings'.


In [ ]:
### File Selection Logic
This part handles the GUI interaction.

In [ ]:
def select_csv_files():
    """
    Opens a file dialog to select multiple CSV files.

    Returns:
        list: A list of selected file paths.
    """
    root = tk.Tk()
    root.attributes('-topmost', True)
    root.withdraw()

    file_paths = filedialog.askopenfilenames(
        title="Select CSV files with job postings",
        filetypes=[("CSV Files", "*.csv")]
    )
    root.destroy()
    return file_paths

file_paths = select_csv_files()

if not file_paths:
    print("No files selected. Operation cancelled.")
else:
    print(f"Selected {len(file_paths)} files.")

### Data Loading and Preprocessing
Here we handle the logic of merging files and cleaning the DataFrame.

In [ ]:
def process_job_postings(file_paths):
    """
    Loads, merges, and cleans job posting data from CSV files.

    Args:
        file_paths (list): List of paths to CSV files.

    Returns:
        pd.DataFrame: A cleaned and deduplicated DataFrame.
    """
    dfs = []
    try:
        # Sort paths to maintain chronological order if filenames contain dates
        for f in sorted(file_paths):
            df = pd.read_csv(f)
            if 'Unnamed: 0' in df.columns:
                df = df.drop(columns=['Unnamed: 0'])
            dfs.append(df)

        merged_df = pd.concat(dfs, ignore_index=True)

        # Standardize column names (lowercase, no spaces, underscores)
        merged_df.columns = merged_df.columns.str.strip().str.lower().str.replace(' ', '_')

        # Deduplicate by ID, keeping the last occurrence (most recent)
        initial_count = len(merged_df)
        merged_df = merged_df.drop_duplicates(subset=['id'], keep='last')
        removed_count = initial_count - len(merged_df)

        print(f"Removed {removed_count} duplicates from the CSV batch.")
        print(f"Total {len(merged_df)} unique rows ready for database merge.")

        return merged_df

    except Exception as e:
        print(f"Error while processing CSV files: {e}")
        return None

if file_paths:
    cleaned_df = process_job_postings(file_paths)

### Database Upsert (Merge)
This cell executes the SQL logic to synchronize the DataFrame with PostgreSQL.

In [ ]:
def upsert_to_postgres(df, db_handler):
    """
    Performs an UPSERT operation into the job_postings table using a temporary table.

    Args:
        df (pd.DataFrame): The cleaned DataFrame to upload.
        db_handler: Database handler object with a SQLAlchemy engine.
    """
    if df is None or df.empty:
        print("DataFrame is empty. Skipping database upload.")
        return

    try:
        with db_handler.engine.begin() as conn:
            # 1. Upload DataFrame to a temporary table
            df.to_sql('tmp_job_postings', con=conn, if_exists='replace', index=False)

            # 2. Execute the UPSERT (Merge) query
            upsert_query = text("""
                INSERT INTO job_postings (
                    id, stanowisko, firma, poziom, kategoria, technologie, lokalizacja,
                    wynagrodzenie_od, wynagrodzenie_do, waluta, utworzono, zaktualizowano, scraped_at, source, data_pobrania
                )
                SELECT
                    tmp.id,
                    tmp.stanowisko,
                    tmp.firma,
                    CAST(tmp.poziom AS senioritylevel),
                    tmp.kategoria,
                    tmp.technologie,
                    tmp.lokalizacja,
                    CAST(tmp.wynagrodzenie_od AS INTEGER),
                    CAST(tmp.wynagrodzenie_do AS INTEGER),
                    CAST(tmp.waluta AS currency),
                    CAST(tmp.utworzono AS TIMESTAMP),
                    CAST(tmp.zaktualizowano AS TIMESTAMP),
                    CAST(tmp.scraped_at AS TIMESTAMP),
                    CAST(tmp.source AS source),
                    CURRENT_TIMESTAMP
                FROM tmp_job_postings tmp
                ON CONFLICT (id) DO UPDATE SET
                    stanowisko = EXCLUDED.stanowisko,
                    firma = EXCLUDED.firma,
                    poziom = EXCLUDED.poziom,
                    kategoria = EXCLUDED.kategoria,
                    technologie = EXCLUDED.technologie,
                    lokalizacja = EXCLUDED.lokalizacja,
                    wynagrodzenie_od = EXCLUDED.wynagrodzenie_od,
                    wynagrodzenie_do = EXCLUDED.wynagrodzenie_do,
                    waluta = EXCLUDED.waluta,
                    utworzono = EXCLUDED.utworzono,
                    zaktualizowano = EXCLUDED.zaktualizowano,
                    scraped_at = EXCLUDED.scraped_at,
                    source = EXCLUDED.source,
                    data_pobrania = EXCLUDED.data_pobrania;
            """)
            conn.execute(upsert_query)

            # 3. Cleanup: Drop the temporary table
            conn.execute(text("DROP TABLE tmp_job_postings;"))

        print(f"Success! {len(df)} rows merged into 'job_postings' table.")

    except Exception as e:
        print(f"Error during database upsert: {e}")

# Assuming 'db' is your database handler instance
if 'cleaned_df' in locals():
    upsert_to_postgres(cleaned_df, db)